In [5]:
import torch
x = torch.arange(12)
x.shape

torch.Size([12])

In [ ]:
X = x.reshape((3,4))
X

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])

In [ ]:
a = torch.tensor([[1,2,3,4],[2,3,4,5],[3,4,5,6]])
b = torch.tensor([[[1,2],[3,4]],[[2,3],[4,5]],[[3,4],[5,6]]])
b

tensor([[[1, 2],
         [3, 4]],

        [[2, 3],
         [4, 5]],

        [[3, 4],
         [5, 6]]])

In [ ]:
# + - * / ** ==  这些都被升级为按元素进行！！！
# 矩阵乘法需要使用 @ 运算符

M = torch.arange(12).reshape(3,4)
N = torch.tensor([[4,3,2,1],[6,5,4,3],[8,7,6,4]])

# 这里的理解，最好不要理解成矩阵的行、列相加
# 使用 cat 函数合并后，就是下面这种访问
# A[0:3, :] == M
# A[3:6, :] == N
# M[i, j] → A[i,     j]
# N[i, j] → A[i + 3, j]
torch.cat((M,N),dim=0),torch.cat((M,N),dim=1)

(tensor([[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11],
         [ 4,  3,  2,  1],
         [ 6,  5,  4,  3],
         [ 8,  7,  6,  4]]),
 tensor([[ 0,  1,  2,  3,  4,  3,  2,  1],
         [ 4,  5,  6,  7,  6,  5,  4,  3],
         [ 8,  9, 10, 11,  8,  7,  6,  4]]))

In [ ]:
a = torch.arange(12)
b = a.reshape((3,4))
# b[:] 等价于 b[:,:]
b[:]=2  # 这里是将标量 2 广播为切片 b[:] 的整个形状 (3,4)
# 这里的核心是 标量 2 被广播的是 (3,4), 而不是 (1,4),
# 因为, 切片不是逐行循环，而是一次得到一个包含所有选中元素的张量
a

tensor([2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [13]:
M == N

tensor([[False, False,  True, False],
        [False,  True, False, False],
        [ True, False, False, False]])

In [ ]:
# sum() 默认会对所有元素求和

In [ ]:
# 广播机制（broadcasting）：适用于支持广播的运算
# 1. 从右向左对齐各轴，缺失的轴按长度 1 处理。
# 2. 每一对对应轴的长度必须相等，或其中一个为 1。
# 3. 长度为 1 的轴按需扩展，通常不实际复制底层数据。

In [ ]:
# 转换为 Numpy 张量
X = torch.arange(12).reshape(3,4)
A = X.numpy()
B = torch.tensor(A)
type(A),type(B)

(numpy.ndarray, torch.Tensor)

In [18]:
# 转换为 python 标量
a = torch.tensor([3.5])
a,a.item(),float(a),int(a)

(tensor([3.5000]), 3.5, 3.5, 3)

## 数据预处理

In [19]:
import os

os.makedirs(os.path.join('..','data'),exist_ok=True)
data_file = os.path.join('..','data','house_tiny.csv')
with open(data_file, 'w') as f:
  f.write('NumRooms,Alley,Price\n') # 列名
  f.write('NA,Pave,127500\n')       # 每行表示一个数据样本
  f.write('2,NA,106000\n')
  f.write('4,NA,178100\n')
  f.write('NA,NA,140000\n')

In [21]:
import pandas as pd

data = pd.read_csv(data_file)
data

,NumRooms,Alley,Price
0,NaN,Pave,127500
1,2.0,NaN,106000
2,4.0,NaN,178100
3,NaN,NaN,140000


In [27]:
# 对于数据的缺失，典型的方法有：插值、删除

## 均值填补
inputs,outputs = data.iloc[:,0:2],data.iloc[:,-1]
inputs = inputs.fillna(inputs.mean(numeric_only=True))  
inputs

,NumRooms,Alley
0,3.0,Pave
1,2.0,NaN
2,4.0,NaN
3,3.0,NaN


In [ ]:
# 独立热编码(one-hot encoding)
# 把类别数据转换称数值指示列(将原来的列替换成新的列,列的名字就是原来的列+类别)
# dummy 在统计学中指 "虚拟变量"或"指示变量", 用 0/1 表示某个类别是否出现
# dummies 是它的复数
"""
常见用法
inputs = pd.get_dummies(
    inputs,
    dummy_na=True,
    dtype=float,
)
传入 DataFrame 时，默认编码 object,string,category 类型的列；
  普通数值列保留，不做编码。
dtype 控制新生成列的类型：默认 bool，可指定 int 或 float；
  不改变原样保留列的类型。     也就是说, dummy 会生成新的列
"""
inputs = pd.get_dummies(inputs,dummy_na=True, dtype=float)
inputs

,NumRooms,Alley_Pave,Alley_nan
0,3.0,1.0,0.0
1,2.0,0.0,1.0
2,4.0,0.0,1.0
3,3.0,0.0,1.0


In [29]:
import torch
X, y = torch.tensor(inputs.values), torch.tensor(outputs.values)
X,y

(tensor([[3., 1., 0.],
         [2., 0., 1.],
         [4., 0., 1.],
         [3., 0., 1.]], dtype=torch.float64),
 tensor([127500, 106000, 178100, 140000]))